# Pipeline completo: Brecha digital ENAHO–ESCALE 2025

Este notebook ejecuta, en orden, el mismo pipeline maestro disponible en la raíz del proyecto. Para correrlo sin abrir Jupyter, basta con ejecutar `python pipeline.py`. La etiqueta `exclusion_digital` es un proxy de no uso declarado de Internet entre quienes asisten; no mide competencias digitales ni rendimiento académico.

Los microdatos y los resultados procesados están ignorados por Git mediante `.gitignore`, para evitar subir archivos pesados o de acceso controlado. Debes tener las fuentes originales disponibles localmente en `data/raw/`.

## Qué ocurre al ejecutar `pipeline.py`

1. Revisa que existan los microdatos crudos ENAHO y ESCALE.
2. Limpia y procesa los archivos, conserva las llaves, filtra edad escolar y une persona, hogar, Sumaria y ESCALE.
3. Valida la tabla integrada y reporta llaves, cobertura y etiquetas disponibles.
4. Explora distribuciones y faltantes antes de modelar.
5. Entrena y compara K-Means y Ward; elige perfiles con métricas de clustering.
6. Separa por conglomerado, compara cuatro clasificadores y evalúa en prueba.
7. Genera gráficos de evaluación y SHAP para el clasificador ganador.

Cada paso muestra qué está haciendo y se detiene si falla. Los notebooks de entrenamiento de cada etapa están separados en esta carpeta.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pipeline.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto ni pipeline.py.")

RAW_ENAHO = PROJECT_ROOT / "data" / "raw" / "enaho"
RAW_ESCALE = PROJECT_ROOT / "data" / "raw" / "minedu"
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Python del kernel: {sys.executable}")
print(f"ENAHO: {RAW_ENAHO}")
print(f"ESCALE: {RAW_ESCALE}")

In [ ]:
required_raw = [
    RAW_ENAHO / "enaho01-2021-2025-100-panel.csv",
    RAW_ENAHO / "enaho01-2021-2025-200-panel.csv",
    RAW_ENAHO / "enaho01a-2021-2025-300-panel.csv",
    RAW_ENAHO / "sumaria-2021-2025-panel.csv",
    RAW_ESCALE / "Padron.dbf",
]
missing_raw = [path for path in required_raw if not path.exists()]
if missing_raw:
    raise FileNotFoundError("Faltan archivos de entrada:\n" + "\n".join(map(str, missing_raw)))
print(f"Entradas requeridas encontradas: {len(required_raw)}")
print("Nota: ejecuta este notebook usando el entorno .venv del proyecto.")

## 1. Integrar datos

Primero se prepara la tabla única de estudiantes; las etapas de modelos dependen de ella. Esta etapa se ejecuta dentro del pipeline maestro.

In [ ]:
import subprocess

subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "pipeline.py")],
    cwd=PROJECT_ROOT,
    check=True,
)

## Revisar integración y cobertura

Confirma filas, clases disponibles y cobertura de uniones antes de modelar.

In [ ]:
import pandas as pd
from IPython.display import display

PROCESSED = PROJECT_ROOT / "data" / "processed"
DATASET_PATH = PROCESSED / "dataset_enaho_escale_2025.parquet"
for path in [DATASET_PATH, PROCESSED / "reporte_integracion_2025.csv", PROCESSED / "diccionario_variables_2025.csv"]:
    if not path.exists():
        raise FileNotFoundError(f"Falta {path}. Ejecuta el pipeline primero.")

data = pd.read_parquet(DATASET_PATH)
print(f"Dimensiones: {data.shape[0]:,} filas × {data.shape[1]} columnas")
print("Etiqueta exclusion_digital:")
display(data["exclusion_digital"].value_counts(dropna=False).rename_axis("clase").to_frame("casos"))
print("Reporte de integración:")
display(pd.read_csv(PROCESSED / "reporte_integracion_2025.csv"))
print("Diccionario de variables:")
display(pd.read_csv(PROCESSED / "diccionario_variables_2025.csv"))

## 2. Entrenar y probar clustering

Se comparan K-Means y Ward para k=2…6 con Silhouette, Davies–Bouldin y Calinski–Harabasz. K-Means y el perfil final usan una muestra reproducible de hasta 5 000 filas; no se usa la etiqueta supervisada para formar los grupos.

In [ ]:
from IPython.display import Image, display

CLUSTER_DIR = PROJECT_ROOT / "informes" / "resultados_clustering"
cluster_metrics_path = CLUSTER_DIR / "metricas_clustering.csv"
profiles_path = CLUSTER_DIR / "perfiles_clusters.csv"
for path in [cluster_metrics_path, profiles_path, CLUSTER_DIR / "clusters_pca.png", CLUSTER_DIR / "silueta_kmeans.png"]:
    if not path.exists():
        raise FileNotFoundError(f"Falta {path}. Ejecuta el pipeline primero.")
display(pd.read_csv(cluster_metrics_path))
display(pd.read_csv(profiles_path))
display(Image(filename=str(CLUSTER_DIR / "clusters_pca.png")))
display(Image(filename=str(CLUSTER_DIR / "silueta_kmeans.png")))

## 3. Entrenar y comparar clasificadores

La Regresión Logística es el baseline; se compara con Random Forest, Extra Trees y XGBoost. GroupKFold valida por conglomerado y el conjunto de prueba queda separado. El script imprime las métricas por modelo al terminar.

In [ ]:
CLASSIFICATION_DIR = PROJECT_ROOT / "informes" / "resultados_clasificacion"
cv_path = CLASSIFICATION_DIR / "comparacion_modelos_groupkfold.csv"
test_path = CLASSIFICATION_DIR / "comparacion_modelos_prueba.csv"
for path in [cv_path, test_path]:
    if not path.exists():
        raise FileNotFoundError(f"Falta {path}. Ejecuta el pipeline primero.")
cv_results = pd.read_csv(cv_path)
test_results = pd.read_csv(test_path)
model_columns = [column for column in ["modelo", "roc_auc_ponderado_mean", "pr_auc_ponderado_mean", "f1_macro_ponderado_mean"] if column in cv_results.columns]
display(cv_results[model_columns].sort_values("roc_auc_ponderado_mean", ascending=False))
display(test_results)
selected_rows = test_results.loc[test_results["seleccionado_por_cv"]]
if len(selected_rows) != 1:
    raise ValueError("Se esperaba exactamente un modelo seleccionado por CV.")
selected_model = selected_rows.iloc[0]["modelo"]
MODEL_DIR = CLASSIFICATION_DIR / selected_model
print(f"Modelo seleccionado por GroupKFold: {selected_model}")
display(pd.read_csv(MODEL_DIR / "metricas_prueba.csv"))

In [ ]:
model_figures = [
    "curva_roc.png",
    "curva_precision_recall.png",
    "curva_calibracion.png",
    "matriz_confusion.png",
    "shap_beeswarm.png",
    "shap_dependencia_lengua_internet.png",
    "shap_dependencia_internet_lengua.png",
    "shap_waterfall_alto_riesgo.png",
    "shap_waterfall_bajo_riesgo.png",
]
for filename in model_figures:
    image_path = MODEL_DIR / filename
    if not image_path.exists():
        raise FileNotFoundError(f"Falta la figura {image_path}.")
    print(filename)
    display(Image(filename=str(image_path)))